# Notebook 11: Parallel BFS, Streaming, and Agent2Agent (A2A)

This notebook is a deep-dive into five advanced Multigen features that were added together as a coherent enterprise execution upgrade:

| Feature | What It Solves |
|---|---|
| **Parallel BFS execution** | Sequential graph traversal is slow for independent nodes |
| **`depends_on` field** | Logical dependencies that live outside the edge topology |
| **Partition-aware fan-out** | Single task queue becomes a bottleneck for large parallel batches |
| **SSE Streaming** | Polling entire workflow state is expensive; clients want push events |
| **Polling fallback** | SSE connections drop; clients need a catch-up mechanism |
| **Agent2Agent (A2A) protocol** | Calling external agents without coupling to their internals |
| **Distributed agent store** | Dynamic agents created on Worker-1 are invisible to Worker-2 |
| **`SyncMultigenClient` in Jupyter** | `asyncio.run()` conflicts with Jupyter's own event loop |

---

## Prerequisites

Before running any cell:

```bash
# Terminal 1 — Temporal development server
temporal server start-dev

# Terminal 2 — Multigen orchestrator
uvicorn orchestrator.main:app --reload --port 8000

# Terminal 3 — Default Temporal worker
python workers/graph_worker.py

# Terminal 4 (optional, for partition demo) — Second worker on different queue
TEMPORAL_TASK_QUEUE=flow-task-queue-b python workers/graph_worker.py

# Terminal 5 (optional) — Third worker
TEMPORAL_TASK_QUEUE=flow-task-queue-c python workers/graph_worker.py
```

MongoDB must be running on `localhost:27017` (default).

---
## Section 0 — Setup & Prerequisites

In [ ]:
# ── Install dependencies (run once) ───────────────────────────────────────────
# Uncomment and execute this cell if you haven't installed the packages yet.
#
# !pip install httpx pydantic requests sseclient-py
#
# The SDK itself is installed in editable mode when you clone the repo:
# !pip install -e .

In [ ]:
import json
import time
import uuid
import pprint
import asyncio
import requests

# ── Multigen SDK ───────────────────────────────────────────────────────────────
# SyncMultigenClient wraps the async client in a background thread+event loop
# so it works correctly from Jupyter without conflicting with the notebook's
# own event loop (IPython runs its own asyncio loop since Jupyter 7).
from sdk.multigen.sync_client import SyncMultigenClient
from sdk.multigen.client import MultigenClient
from sdk.multigen.models import FanOutRequest, FanOutNodeDef

# ── A2A client ─────────────────────────────────────────────────────────────────
from a2a.client import A2AClient

# ── Orchestrator config (env-driven) ───────────────────────────────────────────
import orchestrator.services.config as config

ORCHESTRATOR_URL = "http://localhost:8000"

print("Imports OK")
print(f"Temporal URL        : {config.TEMPORAL_SERVER_URL}")
print(f"Default task queue  : {config.TEMPORAL_TASK_QUEUE}")
print(f"Task queue pool     : {config.TEMPORAL_TASK_QUEUES}")
print(f"MongoDB URI         : {config.MONGODB_URI}")

In [ ]:
# ── Initialise the synchronous client ─────────────────────────────────────────
#
# SyncMultigenClient spins up a dedicated daemon thread with its own event loop.
# Every method call submits a coroutine to that loop via
# asyncio.run_coroutine_threadsafe() and blocks the calling thread until done.
#
# This is the recommended pattern for Jupyter notebooks and plain scripts where
# await / async with is not available at the top level.

client = SyncMultigenClient(ORCHESTRATOR_URL)

reachable = client.ping()
print(f"Orchestrator reachable: {reachable}")

if not reachable:
    print("\nWARNING: orchestrator is not running.")
    print("Start it with: uvicorn orchestrator.main:app --reload --port 8000")
    print("The cells below will show expected output but won't execute live.")

---
## Section 1 — Parallel BFS Execution

### The Diamond DAG Problem

Consider the classic diamond dependency graph:

```
        A  (entry)
       / \
      B   C        ← B and C are INDEPENDENT; A is their only upstream
       \ /
        D  (aggregator)
```

Edges: `A→B`, `A→C`, `B→D`, `C→D`

**Sequential BFS** (old behaviour):
- Iteration 1: Execute A → 30s
- Iteration 2: Execute B → 30s  ← C waits even though it's independent
- Iteration 3: Execute C → 30s
- Iteration 4: Execute D → 30s
- **Total: ~120s**

**Parallel BFS** (current behaviour):
- Wave 1: Execute A → 30s
- Wave 2: Execute B **and** C via `asyncio.gather()` → 30s (wall-clock)
- Wave 3: Execute D → 30s
- **Total: ~90s**

For large enterprise graphs with 10+ parallel branches, the savings compound dramatically. A 5-node M&A research batch that took 150s sequentially now finishes in ~30s (the slowest expert).

### How It Works

Each iteration of the BFS loop calls `_collect_ready_batch()`, which drains the pending queue and separates nodes whose dependencies are ALL present in `context` (ready) from those still waiting (deferred). All ready nodes are dispatched simultaneously via `asyncio.gather()`. Temporal routes each as an independent activity, allowing different worker replicas to handle them truly in parallel.

In [ ]:
# ── Section 1.1: Diamond DAG graph definition ──────────────────────────────────
#
# EchoAgent is a trivial agent that echoes back its params — perfect for demos
# because it completes instantly and doesn't need any API keys.
#
# In a real workflow you'd use PlannerAgent, FinancialAnalystAgent, etc.

diamond_graph = {
    "nodes": [
        {
            "id": "A",
            "agent": "EchoAgent",
            "params": {"step": "entry", "description": "Kick-off node"},
            "timeout": 30,
        },
        {
            "id": "B",
            "agent": "EchoAgent",
            "params": {"step": "left-branch", "description": "Financial research"},
            "timeout": 30,
        },
        {
            "id": "C",
            "agent": "EchoAgent",
            "params": {"step": "right-branch", "description": "Legal research"},
            "timeout": 30,
        },
        {
            "id": "D",
            "agent": "EchoAgent",
            "params": {"step": "aggregator", "description": "Synthesis node"},
            "timeout": 30,
        },
        {
            "id": "E",
            "agent": "EchoAgent",
            "params": {"step": "report", "description": "Final report"},
            "timeout": 30,
        },
    ],
    "edges": [
        {"source": "A", "target": "B"},   # A → B
        {"source": "A", "target": "C"},   # A → C  (parallel branch)
        {"source": "B", "target": "D"},   # B → D
        {"source": "C", "target": "D"},   # C → D
        {"source": "D", "target": "E"},   # D → E
    ],
    "entry": "A",
    "max_cycles": 5,
}

print("Diamond graph defined:")
print("  Nodes:", [n['id'] for n in diamond_graph['nodes']])
print("  Edges:", [(e['source'], e['target']) for e in diamond_graph['edges']])
print("  Entry:", diamond_graph['entry'])
print()
print("Expected execution waves:")
print("  Wave 1 → [A]")
print("  Wave 2 → [B, C]  (parallel — A is in context, both deps satisfied)")
print("  Wave 3 → [D]     (requires B AND C both in context)")
print("  Wave 4 → [E]")

In [ ]:
# ── Section 1.2: Run the diamond graph and time it ─────────────────────────────

t0 = time.time()
resp = client.run_graph(graph_def=diamond_graph, payload={"run": "diamond-demo"})
t_submit = time.time() - t0

workflow_id = resp.instance_id
print(f"Workflow submitted  : {workflow_id}")
print(f"Submission latency  : {t_submit:.2f}s")

In [ ]:
# ── Section 1.3: Poll until complete and inspect node order ───────────────────

import time

t_start = time.time()
for attempt in range(30):   # poll for up to 30s
    state = client.get_state(workflow_id)
    if state.count >= 5:    # all 5 nodes complete
        break
    time.sleep(1)
    print(f"  [{attempt+1:02d}s] nodes complete: {state.count}/5", end="\r")

elapsed = time.time() - t_start
print(f"\nTotal wall-clock time: {elapsed:.1f}s")
print(f"Nodes completed     : {state.count}")
print()

# Print the streaming event log — the server appended an entry per completed node
# in the order they actually finished. B and C should have very close timestamps.
raw = requests.get(f"{ORCHESTRATOR_URL}/workflows/{workflow_id}/events").json()
events = raw.get("events", [])

print("Node completion order (from SSE event log):")
print(f"{'idx':>4}  {'node':>6}  {'agent':>12}  {'confidence':>10}  timestamp")
print("-" * 70)
for ev in events:
    print(f"{ev['index']:>4}  {ev['node_id']:>6}  {ev['agent']:>12}  {ev['confidence']:>10.2f}  {ev['timestamp']}")

In [ ]:
# ── Section 1.4: Verify B and C executed in the same wave ─────────────────────
#
# Because B and C have identical upstream (only A) and EchoAgent completes
# almost instantly, their timestamps will be within milliseconds of each other.
# In a real graph with 30s LLM calls, the gap would still be ~0 wall-clock
# because asyncio.gather() dispatches them simultaneously to Temporal.

from datetime import datetime

ts = {ev["node_id"]: ev["timestamp"] for ev in events}

if "B" in ts and "C" in ts:
    dt_b = datetime.fromisoformat(ts["B"].replace("Z", "+00:00"))
    dt_c = datetime.fromisoformat(ts["C"].replace("Z", "+00:00"))
    gap = abs((dt_b - dt_c).total_seconds())
    print(f"B completed at: {ts['B']}")
    print(f"C completed at: {ts['C']}")
    print(f"Wall-clock gap between B and C: {gap:.3f}s")
    if gap < 2.0:
        print("PASS: B and C executed in the same parallel wave.")
    else:
        print("NOTE: gap > 2s — worker may be single-threaded or under load.")
else:
    print("Event log not yet available — orchestrator may still be running.")

---
## Section 2 — The `depends_on` Field

### Motivation

Graph edges describe **data flow**: node B reads node A's output, so B must run after A. But sometimes two nodes have a **logical dependency** that isn't expressed as data flow:

- A model inference node must wait for both data to be fetched **and** the GPU worker pool to be warmed up — even if the inference node doesn't read the warmup output.
- A rate-limited API call must wait for a rate-limiter node to grant a token.
- A batch processing node must wait for all pre-processing shards to finish, even if it only reads a summary.

Adding artificial edges (e.g. `warmup → inference`) pollutes the edge topology with non-data-flow concerns and can cause confusing dependency chains.

The `depends_on` field lets you declare these logical dependencies **explicitly** on the node, separate from the edge list.

### Implementation

Inside `_deps_satisfied(nid)` in the engine, the check unions **edge-inferred deps** and **explicit `depends_on` deps**:

```python
edge_deps   = [e["source"] for e in _all_edges() if e.get("target") == nid]
explicit    = nd.get("depends_on") or []
all_deps    = set(edge_deps) | set(explicit)
return all(dep in context for dep in all_deps)
```

A node with `depends_on: ["A", "B"]` will not enter the ready batch until both A and B appear in the workflow context, regardless of whether there are matching edges.

In [ ]:
# ── Section 2.1: Graph with explicit depends_on ────────────────────────────────
#
# Scenario: ML inference pipeline
#
#   data_fetch ──────────────────────────────────────────┐
#                                                        ▼
#   model_load   (no edge to inference)          [ inference ]  ──►  [ report ]
#
# data_fetch and model_load are independent — they run in parallel.
# inference has edge dep on data_fetch (reads its output) AND
# an explicit depends_on: ["model_load"] (logical dep — GPU warmup).
#
# Without depends_on, inference could start before model_load finishes,
# causing an "agent not ready" error on the GPU worker.

inference_graph = {
    "nodes": [
        {
            "id": "data_fetch",
            "agent": "EchoAgent",
            "params": {"task": "fetch_training_data", "source": "s3://bucket/data"},
            "timeout": 60,
        },
        {
            "id": "model_load",
            "agent": "EchoAgent",
            "params": {"task": "warm_up_gpu", "model": "llama-3-70b"},
            "timeout": 90,
        },
        {
            "id": "inference",
            "agent": "EchoAgent",
            # Edge dep: reads data from data_fetch  →  expressed as edge below
            # Logical dep: GPU must be warm         →  expressed with depends_on
            "depends_on": ["model_load"],           # <── KEY FIELD
            "params": {
                "task": "run_inference",
                # In a real graph this would reference data_fetch's output:
                # "input_data": "{{steps.data_fetch.output.data}}"
                "input_data": "ref:data_fetch.output",
            },
            "timeout": 120,
        },
        {
            "id": "report",
            "agent": "EchoAgent",
            "params": {"task": "generate_report", "format": "pdf"},
            "timeout": 30,
        },
    ],
    "edges": [
        # data_fetch and model_load both start from 'entry' implicitly
        # (They have no incoming edges — the engine starts them at entry=data_fetch
        # and injects model_load via fan-out or as a parallel entry.)
        #
        # For simplicity in this demo, we chain them:
        {"source": "data_fetch", "target": "inference"},  # data flow edge
        {"source": "inference",  "target": "report"},
    ],
    # model_load runs from the start (same entry wave as data_fetch)
    # by injecting it as a standalone root node:
    "entry": "data_fetch",
    "max_cycles": 5,
}

# Inject model_load as a parallel root by also starting it via fan-out after run_graph
# OR list it as a second entry point via a no-op root node (shown below).

# Cleaner pattern: use a coordinator entry node
inference_graph_clean = {
    "nodes": [
        {
            "id": "start",
            "agent": "EchoAgent",
            "params": {"task": "coordinator"},
        },
        {
            "id": "data_fetch",
            "agent": "EchoAgent",
            "params": {"task": "fetch_training_data", "source": "s3://bucket/data"},
            "timeout": 60,
        },
        {
            "id": "model_load",
            "agent": "EchoAgent",
            "params": {"task": "warm_up_gpu", "model": "llama-3-70b"},
            "timeout": 90,
        },
        {
            "id": "inference",
            "agent": "EchoAgent",
            # Edge dep from data_fetch (data flow)
            # Explicit dep on model_load (logical / resource dep)
            "depends_on": ["model_load"],
            "params": {
                "task": "run_inference",
                "input_data": "ref:data_fetch.output",
            },
            "timeout": 120,
        },
        {
            "id": "report",
            "agent": "EchoAgent",
            "params": {"task": "generate_report", "format": "pdf"},
            "timeout": 30,
        },
    ],
    "edges": [
        {"source": "start",      "target": "data_fetch"},   # start kicks off data_fetch
        {"source": "start",      "target": "model_load"},   # and model_load in parallel
        {"source": "data_fetch", "target": "inference"},    # data flow
        {"source": "inference",  "target": "report"},
    ],
    "entry": "start",
    "max_cycles": 5,
}

print("inference_graph_clean nodes:")
for n in inference_graph_clean["nodes"]:
    deps_on = n.get("depends_on", [])
    edge_targets = [e["source"] for e in inference_graph_clean["edges"] if e["target"] == n["id"]]
    print(f"  {n['id']:15s}  edge_deps={edge_targets}  depends_on={deps_on}")

In [ ]:
# ── Section 2.2: Run the depends_on graph ─────────────────────────────────────

resp2 = client.run_graph(graph_def=inference_graph_clean, payload={"demo": "depends_on"})
wid2 = resp2.instance_id
print(f"Workflow ID: {wid2}")

# Poll until complete
for _ in range(30):
    state2 = client.get_state(wid2)
    if state2.count >= 5:
        break
    time.sleep(1)

raw2 = requests.get(f"{ORCHESTRATOR_URL}/workflows/{wid2}/events").json()
events2 = raw2.get("events", [])

print("\nNode completion order:")
for ev in events2:
    print(f"  [{ev['index']}] {ev['node_id']:15s} at {ev['timestamp']}")

# Verify inference completed AFTER both data_fetch AND model_load
order = [ev["node_id"] for ev in events2]
if "inference" in order and "model_load" in order and "data_fetch" in order:
    idx_inf   = order.index("inference")
    idx_ml    = order.index("model_load")
    idx_df    = order.index("data_fetch")
    passed = idx_inf > idx_ml and idx_inf > idx_df
    status = "PASS" if passed else "FAIL"
    print(f"\n{status}: inference (index={idx_inf}) ran after "
          f"data_fetch (index={idx_df}) and model_load (index={idx_ml})")

### Why `depends_on` vs. An Extra Edge?

You could add an edge `model_load → inference` to get the same sequencing. The difference is semantic and operational:

| Approach | Meaning | Drawback |
|---|---|---|
| Extra edge | "inference reads model_load's output" | Confusing — inference doesn't use model_load's output. Forces `_resolve_refs` to look for a `steps.model_load.*` reference that doesn't exist. |
| `depends_on` | "inference cannot start until model_load is in context" | Clean separation of data-flow edges from resource/sequencing constraints. |

Use `depends_on` for: rate limiting, resource pre-warming, shard barriers, human approval gates that aren't part of the data pipeline.

---
## Section 3 — Partition-Aware Fan-Out

### The Single-Queue Bottleneck

When Multigen dispatches 10 nodes simultaneously via `asyncio.gather()`, Temporal submits 10 activities to the task queue. If only one worker process is listening on that queue, the activities execute sequentially (the worker can only process one at a time). The wall-clock time collapses back to sequential.

**Partition-aware fan-out** solves this by distributing activities across **multiple task queues**, each served by a dedicated worker process (or pool). Temporal routes each activity to the appropriate worker based on the task queue specified in the activity options.

### Configuration

Set the `TEMPORAL_TASK_QUEUES` environment variable to a comma-separated list of queue names:

```bash
export TEMPORAL_TASK_QUEUES=flow-task-queue,flow-task-queue-b,flow-task-queue-c
```

Start a worker for each queue:

```bash
TEMPORAL_TASK_QUEUE=flow-task-queue   python workers/graph_worker.py
TEMPORAL_TASK_QUEUE=flow-task-queue-b python workers/graph_worker.py
TEMPORAL_TASK_QUEUE=flow-task-queue-c python workers/graph_worker.py
```

### Fan-Out Spec with `task_queues`

Add a `task_queues` list to any fan-out spec. The engine assigns queues round-robin across fan-out nodes:

```json
{
  "group_id": "expert_panel",
  "consensus": "highest_confidence",
  "task_queues": ["flow-task-queue", "flow-task-queue-b", "flow-task-queue-c"],
  "nodes": [
    {"id": "financial", "agent": "EchoAgent"},
    {"id": "legal",     "agent": "EchoAgent"},
    {"id": "market",    "agent": "EchoAgent"}
  ]
}
```

Round-robin assignment:
- `financial` → `flow-task-queue`   (index 0 % 3 = 0)
- `legal`     → `flow-task-queue-b` (index 1 % 3 = 1)
- `market`    → `flow-task-queue-c` (index 2 % 3 = 2)

Each expert runs on a different worker process in true multi-process parallelism.

In [ ]:
# ── Section 3.1: Inspect the TEMPORAL_TASK_QUEUES config ──────────────────────

import orchestrator.services.config as config

print("Current task queue configuration:")
print(f"  TEMPORAL_TASK_QUEUE  (default): {config.TEMPORAL_TASK_QUEUE}")
print(f"  TEMPORAL_TASK_QUEUES (pool)   : {config.TEMPORAL_TASK_QUEUES}")
print()

if len(config.TEMPORAL_TASK_QUEUES) == 1:
    print("TIP: Only one queue is configured.")
    print("Set TEMPORAL_TASK_QUEUES=queue-a,queue-b,queue-c to enable partition routing.")
    print("Each queue needs a worker: TEMPORAL_TASK_QUEUE=queue-b python workers/graph_worker.py")
else:
    print(f"Partition routing ENABLED across {len(config.TEMPORAL_TASK_QUEUES)} queues:")
    for i, q in enumerate(config.TEMPORAL_TASK_QUEUES):
        print(f"  [{i}] {q}")

In [ ]:
# ── Section 3.2: Partition-aware fan-out via API ───────────────────────────────
#
# Start a minimal entry graph, then signal a fan-out with task_queues.

entry_graph = {
    "nodes": [
        {
            "id": "coordinator",
            "agent": "EchoAgent",
            "params": {"role": "dispatch_experts"},
        }
    ],
    "edges": [],
    "entry": "coordinator",
    "max_cycles": 5,
}

resp3 = client.run_graph(graph_def=entry_graph)
wid3 = resp3.instance_id
print(f"Workflow: {wid3}")

# Wait for coordinator to complete
time.sleep(3)

# Signal a partition-aware fan-out with 3 expert nodes across 3 queues
fan_out_spec = {
    "group_id": "ma_expert_panel",
    "consensus": "highest_confidence",
    "task_queues": [
        "flow-task-queue",      # Worker pool A
        "flow-task-queue-b",    # Worker pool B
        "flow-task-queue-c",    # Worker pool C
    ],
    "nodes": [
        {
            "id": "financial_expert",
            "agent": "EchoAgent",
            "params": {"domain": "financial", "company": "TargetCorp"},
            # task_queue will be injected as "flow-task-queue" (index 0 % 3)
        },
        {
            "id": "legal_expert",
            "agent": "EchoAgent",
            "params": {"domain": "legal", "company": "TargetCorp"},
            # task_queue will be injected as "flow-task-queue-b" (index 1 % 3)
        },
        {
            "id": "market_expert",
            "agent": "EchoAgent",
            "params": {"domain": "market", "company": "TargetCorp"},
            # task_queue will be injected as "flow-task-queue-c" (index 2 % 3)
        },
    ],
}

from sdk.multigen.models import FanOutRequest, FanOutNodeDef

fan_req = FanOutRequest(
    group_id=fan_out_spec["group_id"],
    consensus=fan_out_spec["consensus"],
    nodes=[
        FanOutNodeDef(
            id=n["id"],
            agent=n["agent"],
            params=n["params"],
        )
        for n in fan_out_spec["nodes"]
    ],
)

# For partition routing, pass task_queues in the raw request body via requests
# (the SDK FanOutRequest does not yet model task_queues — use raw HTTP)
raw_fan_body = fan_out_spec.copy()
resp_fan = requests.post(
    f"{ORCHESTRATOR_URL}/workflows/{wid3}/fan-out",
    json=raw_fan_body,
)
print(f"Fan-out signal: {resp_fan.status_code}")
if resp_fan.ok:
    print(json.dumps(resp_fan.json(), indent=2))

In [ ]:
# ── Section 3.3: Round-robin queue assignment (conceptual) ─────────────────────
#
# This cell shows the actual engine logic for partition assignment without
# running a workflow. Useful to understand the routing algorithm.

def simulate_partition_assignment(nodes, task_queues):
    """Replicate the engine's partition-aware round-robin assignment."""
    result = []
    for idx, node in enumerate(nodes):
        # Only inject if the node doesn't already pin a queue
        if not node.get("task_queue"):
            assigned_queue = task_queues[idx % len(task_queues)]
            node = {**node, "task_queue": assigned_queue}
        result.append(node)
    return result

example_nodes = [
    {"id": "financial", "agent": "EchoAgent"},
    {"id": "legal",     "agent": "EchoAgent"},
    {"id": "market",    "agent": "EchoAgent"},
    {"id": "tech",      "agent": "EchoAgent"},
    {"id": "hr",        "agent": "EchoAgent", "task_queue": "special-queue"},  # pinned
]

queues = ["flow-task-queue", "flow-task-queue-b", "flow-task-queue-c"]
assigned = simulate_partition_assignment(example_nodes, queues)

print("Partition-aware assignment (round-robin):")
print(f"  {'node':15s}  {'task_queue':25s}  pinned?")
print("-" * 55)
for n in assigned:
    pinned = "yes (pre-set)" if n["id"] == "hr" else ""
    print(f"  {n['id']:15s}  {n['task_queue']:25s}  {pinned}")

---
## Section 4 — SSE Streaming (Real-time Node Completion)

### Why Streaming?

Polling `GET /workflows/{id}/state` returns the entire MongoDB snapshot on every request. For a 50-node workflow running over 10 minutes, this is expensive and gives coarse granularity. Instead, the engine maintains an **append-only streaming event log** (`self._completed_events`) that grows by one entry per completed node. Clients can:

1. **SSE** — open `GET /workflows/{id}/stream` and receive push events as they happen
2. **Polling fallback** — `GET /workflows/{id}/events?since=N` to fetch only new events (since index N)

### Event Format

Each event is a JSON object:

```json
{
  "index":      4,
  "node_id":    "legal_expert",
  "agent":      "EchoAgent",
  "confidence": 0.85,
  "timestamp":  "2026-03-21T14:32:11.004Z",
  "output":     {"echo": {"domain": "legal", "company": "TargetCorp"}}
}
```

### Reconnect with `since=N`

The client tracks the last `index` it received. On reconnect it requests `?since=N` to receive only events after index N, avoiding re-processing duplicates.

In [ ]:
# ── Section 4.1: SSE stream helper ────────────────────────────────────────────
#
# Opens an SSE connection and prints events as they arrive.
# Runs in a background thread (Jupyter cells are synchronous).

import threading

def stream_workflow(workflow_id: str, max_events: int = 20, timeout: int = 60):
    """
    Open an SSE connection to the Multigen orchestrator and print node
    completion events as they arrive.

    Parameters
    ----------
    workflow_id : str
        Multigen workflow instance ID.
    max_events : int
        Stop after receiving this many events.
    timeout : int
        Maximum seconds to wait for events.
    """
    url = f"{ORCHESTRATOR_URL}/workflows/{workflow_id}/stream"
    print(f"Opening SSE stream: {url}")
    print(f"{'idx':>4}  {'node':>20}  {'confidence':>10}  output_preview")
    print("-" * 70)

    received = 0
    last_event_data = ""

    try:
        with requests.get(url, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            for raw_line in r.iter_lines(decode_unicode=True):
                if not raw_line:
                    # Blank line = end of SSE event block
                    if last_event_data:
                        try:
                            ev = json.loads(last_event_data)
                            output_preview = str(ev.get("output", {}))[:40]
                            print(
                                f"{ev['index']:>4}  "
                                f"{ev['node_id']:>20}  "
                                f"{ev['confidence']:>10.2f}  "
                                f"{output_preview}"
                            )
                            received += 1
                            last_event_data = ""
                            if received >= max_events:
                                print(f"[Stopped after {received} events]")
                                break
                        except json.JSONDecodeError:
                            last_event_data = ""
                    continue

                if raw_line.startswith("data: "):
                    last_event_data = raw_line[6:]  # strip "data: " prefix
                elif raw_line.startswith("event: "):
                    pass  # event type (node_complete, workflow_done, etc.)
                elif raw_line.startswith(": "):
                    pass  # SSE comment / heartbeat

    except requests.exceptions.ConnectionError:
        print("[SSE connection closed by server — workflow may have completed]")
    except Exception as exc:
        print(f"[SSE error: {exc}]")

    print(f"\nTotal events received: {received}")


print("stream_workflow() helper defined.")
print("Usage: stream_workflow(workflow_id, max_events=10)")

In [ ]:
# ── Section 4.2: Start a workflow and open an SSE stream ──────────────────────
#
# We start the stream in a background thread so the notebook cell doesn't block.
# Then we submit the workflow — events flow in real time.

stream_graph = {
    "nodes": [
        {"id": "alpha",   "agent": "EchoAgent", "params": {"order": 1}},
        {"id": "beta",    "agent": "EchoAgent", "params": {"order": 2}},
        {"id": "gamma",   "agent": "EchoAgent", "params": {"order": 3}},
        {"id": "delta",   "agent": "EchoAgent", "params": {"order": 4}},
    ],
    "edges": [
        {"source": "alpha", "target": "beta"},
        {"source": "alpha", "target": "gamma"},   # beta+gamma run in parallel
        {"source": "beta",  "target": "delta"},
        {"source": "gamma", "target": "delta"},
    ],
    "entry": "alpha",
    "max_cycles": 5,
}

resp4 = client.run_graph(graph_def=stream_graph)
wid4 = resp4.instance_id
print(f"Workflow: {wid4}")

# Open SSE stream in a background thread
t = threading.Thread(target=stream_workflow, args=(wid4, 10, 30), daemon=True)
t.start()

# Wait for streaming thread to finish
t.join(timeout=35)
print("\nSSE stream closed.")

In [ ]:
# ── Section 4.3: Polling fallback with ?since=N ───────────────────────────────
#
# If an SSE connection drops, the client tracks its last seen index and
# resumes by polling GET /workflows/{id}/events?since=N.
#
# This endpoint returns only events with index > N, so the client can
# reconstruct the full stream without re-processing old events.

print("Polling fallback demonstration:")
print(f"Workflow: {wid4}")
print()

# Simulate: we already received events 0 and 1, now catch up from index 1
since = 1
url_poll = f"{ORCHESTRATOR_URL}/workflows/{wid4}/events?since={since}"
print(f"GET {url_poll}")

r_poll = requests.get(url_poll)
if r_poll.ok:
    data = r_poll.json()
    events_since = data.get("events", [])
    total = data.get("total", 0)
    print(f"Total events in log: {total}")
    print(f"New events (since index {since}): {len(events_since)}")
    print()
    for ev in events_since:
        print(f"  [{ev['index']}] {ev['node_id']:15s}  confidence={ev['confidence']:.2f}")
else:
    print(f"Error: {r_poll.status_code} — {r_poll.text}")

In [ ]:
# ── Section 4.4: Robust polling loop with reconnect logic ─────────────────────
#
# Production pattern: try SSE first, fall back to polling on connection error,
# track last_index across reconnects to avoid duplicate processing.

def robust_event_consumer(workflow_id: str, callback, max_wait: int = 60):
    """
    Consume workflow completion events with automatic reconnect.

    Tries SSE first. If the connection drops, falls back to polling
    every 2 seconds from the last seen index. Continues until the
    workflow_done sentinel is received or max_wait is exceeded.

    callback: fn(event_dict) called for each new event.
    """
    last_index = -1
    deadline = time.time() + max_wait

    while time.time() < deadline:
        # ── Try SSE ────────────────────────────────────────────────────────
        try:
            url = f"{ORCHESTRATOR_URL}/workflows/{workflow_id}/stream"
            with requests.get(url, stream=True, timeout=10) as r:
                r.raise_for_status()
                buffer = ""
                for line in r.iter_lines(decode_unicode=True):
                    if line.startswith("data: "):
                        buffer = line[6:]
                    elif not line and buffer:
                        try:
                            ev = json.loads(buffer)
                            if ev.get("node_id") == "__workflow_done__":
                                return  # clean exit
                            if ev.get("index", -1) > last_index:
                                last_index = ev["index"]
                                callback(ev)
                        except json.JSONDecodeError:
                            pass
                        buffer = ""
            return  # server closed connection cleanly → workflow done

        except Exception:
            pass  # fall through to polling

        # ── Polling fallback ────────────────────────────────────────────────
        try:
            since = last_index  # only fetch events after the last we saw
            r_p = requests.get(
                f"{ORCHESTRATOR_URL}/workflows/{workflow_id}/events",
                params={"since": since},
                timeout=5,
            )
            if r_p.ok:
                for ev in r_p.json().get("events", []):
                    if ev["index"] > last_index:
                        last_index = ev["index"]
                        callback(ev)
        except Exception:
            pass

        time.sleep(2)


# Demo: consume all events from wid4 using the robust consumer
received_events = []

def on_event(ev):
    received_events.append(ev)
    print(f"  Event [{ev['index']}] {ev['node_id']:15s}  confidence={ev['confidence']:.2f}")

print(f"Consuming events for workflow {wid4} with reconnect logic:")
robust_event_consumer(wid4, callback=on_event, max_wait=30)
print(f"Total events consumed: {len(received_events)}")

---
## Section 5 — Agent2Agent (A2A) Protocol

### What is A2A?

The [Agent2Agent protocol](https://google.github.io/A2A/) is an open standard for interoperability between different AI agent frameworks. It defines:

- **Agent Card** (`/.well-known/agent.json`) — a JSON document advertising the agent's capabilities, supported skills, and authentication requirements
- **JSON-RPC 2.0 task endpoint** (`POST /a2a`) — accepts `tasks/send`, `tasks/get`, `tasks/cancel` methods
- **SSE streaming** (`POST /a2a/stream`) — `tasks/sendSubscribe` for real-time state transitions

Multigen implements **both sides** of A2A:

- **As a server**: exposes all registered agents as A2A skills via `/.well-known/agent.json` and `POST /a2a`
- **As a client**: graph nodes can call remote A2A agents by setting `a2a_endpoint` and `a2a_skill` on a node definition

### Graph Node with `a2a_endpoint`

```json
{
  "id": "remote_risk",
  "a2a_endpoint": "https://partner-risk-system.example.com",
  "a2a_skill": "risk_analysis",
  "params": {"company": "NovaSemi", "scope": "full"},
  "timeout": 120
}
```

The engine detects `a2a_endpoint` and routes the node to `A2AClient.send_task()` instead of the local agent registry.

In [ ]:
# ── Section 5.1: Discover Multigen as an A2A server ───────────────────────────
#
# Any external agent (LangGraph, CrewAI, Vertex AI) would start here:
# fetch /.well-known/agent.json to discover what skills Multigen offers.

agent_card_url = f"{ORCHESTRATOR_URL}/.well-known/agent.json"
print(f"Fetching Agent Card from: {agent_card_url}")
print()

r_card = requests.get(agent_card_url)
if r_card.ok:
    card = r_card.json()
    print(f"Agent name   : {card.get('name')}")
    print(f"Version      : {card.get('version')}")
    print(f"Description  : {card.get('description', '')[:80]}")
    print(f"URL          : {card.get('url')}")
    print(f"Streaming    : {card.get('capabilities', {}).get('streaming')}")
    print()
    print(f"Available skills ({len(card.get('skills', []))}):'")
    for skill in card.get("skills", [])[:10]:
        print(f"  [{skill['id']:30s}]  {skill['name']:20s}  tags={skill.get('tags', [])}")
else:
    print(f"Error: {r_card.status_code}")
    print("Make sure the orchestrator is running.")

In [ ]:
# ── Section 5.2: Call Multigen as an A2A server (external client simulation) ───
#
# Simulate what a LangGraph agent or Vertex AI agent would do to call
# Multigen's EchoAgent via the A2A protocol.

import uuid

task_id = str(uuid.uuid4())
a2a_body = {
    "jsonrpc": "2.0",
    "id": task_id,
    "method": "tasks/send",
    "params": {
        "id": task_id,
        "message": {
            "role": "user",
            "parts": [
                {"type": "text",  "text": "Run EchoAgent on this data"},
                {"type": "data",  "data": {"company": "NovaSemi", "scope": "risk"}},
            ]
        },
        "metadata": {
            "skill_id": "EchoAgent"   # maps to registered agent name
        }
    }
}

print(f"Sending A2A task {task_id[:8]}... to POST /a2a")
r_a2a = requests.post(f"{ORCHESTRATOR_URL}/a2a", json=a2a_body)

if r_a2a.ok:
    body = r_a2a.json()
    print(f"JSON-RPC response id: {body.get('id')}")
    result = body.get("result", {})
    status = result.get("status", {})
    print(f"Task state  : {status.get('state')}")
    print(f"Timestamp   : {status.get('timestamp')}")
    artifacts = result.get("artifacts", [])
    for art in artifacts:
        meta = art.get("metadata", {})
        print(f"Artifact    : name={art.get('name')}  confidence={meta.get('confidence')}")
        for part in art.get("parts", []):
            if part["type"] == "text":
                print(f"  text: {part['text'][:100]}")
else:
    print(f"Error: {r_a2a.status_code} — {r_a2a.text}")

In [ ]:
# ── Section 5.3: Using SyncMultigenClient.a2a_send() ──────────────────────────
#
# The SDK provides a convenience method that wraps the JSON-RPC 2.0 protocol.
# Useful for testing A2A interoperability without the full workflow layer.

print("Calling EchoAgent via client.a2a_send():")
result = client.a2a_send(
    skill_id="EchoAgent",
    text="Analyse company NovaSemi",
    data={"company": "NovaSemi", "domain": "semiconductor"},
)
print(f"Result type: {type(result)}")
pprint.pprint(result)

In [ ]:
# ── Section 5.4: Using A2AClient (async) ──────────────────────────────────────
#
# A2AClient is the async client for calling REMOTE A2A agents from
# within Multigen graph nodes (or directly from async code).
#
# Graph node definition example:
#   {
#     "id": "remote_risk",
#     "a2a_endpoint": "https://partner-system.example.com",
#     "a2a_skill": "risk_analysis",
#     "params": {"company": "NovaSemi"},
#     "timeout": 120
#   }
#
# In Jupyter we can call it using run_until_complete on a fresh event loop.

async def demo_a2a_client():
    """
    Demonstrate A2AClient calling the local Multigen instance as if it
    were a remote partner agent.
    """
    async with A2AClient(ORCHESTRATOR_URL) as a2a:
        # Step 1: Discover the remote agent's capabilities
        card = await a2a.get_agent_card()
        print(f"Remote agent    : {card.name} v{card.version}")
        print(f"Skills available: {[s.id for s in card.skills[:5]]}")
        print()

        # Step 2: Send a task
        result = await a2a.send_task(
            skill_id="EchoAgent",
            text="Due diligence request",
            data={"target": "NovaSemi", "analysis_type": "full"},
        )
        print("Task result:")
        pprint.pprint(result)

# Run in the background thread's event loop via the sync client's executor
# (avoids conflict with Jupyter's own event loop)
import asyncio
import concurrent.futures

_tmp_loop = asyncio.new_event_loop()
_tmp_executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
_tmp_executor.submit(_tmp_loop.run_forever)

future = asyncio.run_coroutine_threadsafe(demo_a2a_client(), _tmp_loop)
future.result(timeout=30)

_tmp_loop.call_soon_threadsafe(_tmp_loop.stop)
_tmp_executor.shutdown(wait=True)

In [ ]:
# ── Section 5.5: Graph node with a2a_endpoint field ───────────────────────────
#
# This shows how a graph definition uses a2a_endpoint / a2a_skill to call
# a remote partner system. The engine detects these fields and routes the
# node to A2AClient.send_task() instead of the local agent registry.

# Example: M&A due diligence calling a specialist partner risk system
hybrid_graph = {
    "nodes": [
        {
            "id": "local_research",
            "agent": "EchoAgent",              # uses local agent
            "params": {"company": "NovaSemi"},
        },
        {
            "id": "remote_risk",
            # a2a_endpoint: calls external partner system
            # (using localhost here since we don't have a real partner system)
            "a2a_endpoint": ORCHESTRATOR_URL,
            "a2a_skill": "EchoAgent",
            "params": {"company": "NovaSemi", "domain": "risk"},
            "timeout": 60,
        },
        {
            "id": "synthesis",
            "agent": "EchoAgent",
            "params": {"task": "synthesise_local_and_remote"},
            "depends_on": ["local_research", "remote_risk"],
        },
    ],
    "edges": [
        {"source": "local_research", "target": "synthesis"},
        {"source": "remote_risk",    "target": "synthesis"},
    ],
    "entry": "local_research",
    "max_cycles": 5,
}

print("Hybrid graph (local + remote A2A nodes):")
for n in hybrid_graph["nodes"]:
    if "a2a_endpoint" in n:
        print(f"  {n['id']:20s}  REMOTE A2A → {n['a2a_endpoint']}  skill={n['a2a_skill']}")
    else:
        print(f"  {n['id']:20s}  LOCAL      → agent={n.get('agent')}")

print()
print("NOTE: The engine routes 'remote_risk' to A2AClient.send_task() automatically")
print("      because 'a2a_endpoint' is present on the node definition.")

---
## Section 6 — Distributed Agent Store (Cross-Worker Hydration)

### The Cross-Worker Problem

Dynamic agents (created at workflow runtime from blueprint specs) are registered in the **in-memory agent registry** of whichever worker ran `create_agent_activity`. Temporal load-balances activities across workers, so subsequent activities for that agent may be routed to a **different worker** — one that has never seen the agent.

```
BEFORE distributed store (broken):

  Worker-1                         Worker-2
  ────────────────────────         ────────────────────────
  create_agent_activity            (idle)
    → registers DynAgent           
    → DynAgent in local registry
                                   agent_activity(DynAgent)
                                     → KeyError: 'DynAgent'
                                     → CRASH
```

### The Solution: MongoDB Agent Spec Store

After creating a dynamic agent locally, `create_agent_activity` persists the blueprint spec to MongoDB. When `agent_activity` runs on any worker and doesn't find the agent in the local registry, it **lazily hydrates** it from MongoDB:

```
AFTER distributed store (working):

  Worker-1                         MongoDB               Worker-2
  ────────────────────────         ─────────────         ────────────────────────
  create_agent_activity                                  (idle)
    → registers DynAgent            
    → store_agent_spec()  ────────► agent_specs coll
                                   { name: DynAgent,
                                     blueprint: {...},
                                     workflow_id: ...,
                                     expires_at: +24h }
                                                         agent_activity(DynAgent)
                                                           → not in local registry
                                                           → hydrate_agent_if_missing()
                                   agent_specs coll ─────► fetch_agent_spec()
                                                           → blueprint retrieved
                                                           → create_agent_from_blueprint()
                                                           → DynAgent cached locally
                                                           → run() succeeds
```

### Key Functions

| Function | Location | Purpose |
|---|---|---|
| `store_agent_spec(name, blueprint, wf_id)` | `flow_engine/graph/agent_store.py` | Write blueprint to MongoDB (upsert) |
| `fetch_agent_spec(name)` | same | Read blueprint; returns None if expired |
| `delete_agent_spec(name)` | same | Called at workflow cleanup |
| `hydrate_agent_if_missing(name)` | same | Check local registry → fetch → recreate |

Specs expire after 24 hours (configurable via `_SPEC_TTL_HOURS`). This prevents MongoDB from accumulating stale specs from old workflow runs.

### MongoDB Schema

Collection: `multigen.agent_specs`

```json
{
  "agent_name":  "DynAgent_abc123",
  "blueprint":   {"system_prompt": "...", "instruction": "..."},
  "workflow_id": "workflow-uuid",
  "created_at":  "2026-03-21T14:00:00Z",
  "expires_at":  "2026-03-22T14:00:00Z"
}
```

In [ ]:
# ── Section 6.1: Inspect the agent_store module ────────────────────────────────
#
# These functions are called by the engine internally.
# We show them here for educational purposes — in production you don't call
# them directly; the engine handles hydration transparently.

from flow_engine.graph.agent_store import (
    store_agent_spec,
    fetch_agent_spec,
    hydrate_agent_if_missing,
    delete_agent_spec,
)

print("agent_store functions imported:")
print(f"  store_agent_spec       : {store_agent_spec.__doc__.strip().splitlines()[0]}")
print(f"  fetch_agent_spec       : {fetch_agent_spec.__doc__.strip().splitlines()[0]}")
print(f"  hydrate_agent_if_missing: {hydrate_agent_if_missing.__doc__.strip().splitlines()[0]}")
print(f"  delete_agent_spec      : {delete_agent_spec.__doc__.strip().splitlines()[0]}")
print()
print("These are async functions called by the Temporal activity layer.")
print("The engine calls hydrate_agent_if_missing() before every agent_activity()")
print("to ensure cross-worker dynamic agent availability.")

In [ ]:
# ── Section 6.2: hydrate_agent_if_missing flow (conceptual simulation) ─────────
#
# We can't call the async store functions directly from a notebook cell without
# a running MongoDB. This cell shows the logic as a pure Python simulation.

# Simulated in-memory stand-ins for the real MongoDB store and local registry
_sim_mongo_store = {}      # simulates MongoDB agent_specs collection
_sim_local_registry = {}   # simulates the in-memory agent registry

def sim_store_agent_spec(agent_name, blueprint, workflow_id):
    """Simulated store_agent_spec — writes to in-memory store."""
    _sim_mongo_store[agent_name] = {"blueprint": blueprint, "workflow_id": workflow_id}
    print(f"  [Worker-1] Stored spec for '{agent_name}' in distributed store")

def sim_fetch_agent_spec(agent_name):
    """Simulated fetch_agent_spec — reads from in-memory store."""
    return _sim_mongo_store.get(agent_name, {}).get("blueprint")

def sim_create_from_blueprint(agent_name, blueprint):
    """Simulated create_agent_from_blueprint — adds to local registry."""
    _sim_local_registry[agent_name] = blueprint
    print(f"  [Worker-2] Reconstructed '{agent_name}' from blueprint, cached locally")

def sim_hydrate_if_missing(agent_name):
    """Simulated hydrate_agent_if_missing."""
    if agent_name in _sim_local_registry:
        print(f"  [Worker-2] '{agent_name}' already in local registry — skip hydration")
        return True
    blueprint = sim_fetch_agent_spec(agent_name)
    if not blueprint:
        print(f"  [Worker-2] '{agent_name}' not found in distributed store — FAIL")
        return False
    sim_create_from_blueprint(agent_name, blueprint)
    return True


# Simulate the cross-worker lifecycle:
print("=" * 60)
print("SIMULATING: Dynamic agent cross-worker lifecycle")
print("=" * 60)
print()
print("Step 1: Worker-1 runs create_agent_activity")
sim_blueprint = {"system_prompt": "You are a specialist M&A analyst.", "instruction": "Analyse NovaSemi"}
sim_store_agent_spec("DynAgent_NovaSemi", sim_blueprint, "workflow-abc123")
print()

print("Step 2: Temporal routes agent_activity to Worker-2 (load balancing)")
print("Step 3: Worker-2 calls hydrate_agent_if_missing before running the agent")
success = sim_hydrate_if_missing("DynAgent_NovaSemi")
print(f"  Hydration result: {'OK' if success else 'FAIL'}")
print()

print("Step 4: Subsequent calls on Worker-2 skip hydration (cached locally)")
sim_hydrate_if_missing("DynAgent_NovaSemi")
print()

print(f"Final local registry on Worker-2: {list(_sim_local_registry.keys())}")

---
## Section 7 — End-to-End Enterprise Example: M&A Due Diligence

This section combines **all seven features** in a single realistic workflow:

- 3 parallel research nodes (financial, legal, market) using **partition-aware fan-out**
- A synthesis node with **`depends_on`** to ensure all research branches complete
- **SSE streaming** to watch completion events in real time
- An epistemic transparency report at the end

### Workflow DAG

```
  [kickoff]
      │
      ├──────────────────────────────────────────────┐
      │                                              │
 [financial_research]  [legal_research]  [market_research]
  (queue-a)             (queue-b)          (queue-c)
      │                    │                    │
      └──────────────┬─────┘────────────────────┘
                     │ depends_on: [financial, legal, market]
                [synthesis]
                     │
               [final_report]
```

In [ ]:
# ── Section 7.1: Define the M&A due diligence graph ───────────────────────────

target_company = "NovaSemi"

ma_graph = {
    "nodes": [
        # ── Entry: kick-off coordinator ───────────────────────────────────
        {
            "id": "kickoff",
            "agent": "EchoAgent",
            "params": {
                "task": "Due diligence kickoff",
                "target": target_company,
                "analyst_team": "M&A Advisory Group",
                "mandate": "Full scope — financial, legal, market",
            },
            "timeout": 30,
        },

        # ── Parallel research nodes (run simultaneously) ───────────────────
        {
            "id": "financial_research",
            "agent": "EchoAgent",
            "params": {
                "domain": "financial",
                "target": target_company,
                "focus": "Revenue growth, EBITDA margins, debt covenants, working capital",
                "years": [2022, 2023, 2024],
            },
            "timeout": 120,
            # task_queue assigned by partition-aware fan-out (not set here)
        },
        {
            "id": "legal_research",
            "agent": "EchoAgent",
            "params": {
                "domain": "legal",
                "target": target_company,
                "focus": "IP ownership, pending litigation, regulatory filings, contracts",
                "jurisdictions": ["US", "EU"],
            },
            "timeout": 120,
        },
        {
            "id": "market_research",
            "agent": "EchoAgent",
            "params": {
                "domain": "market",
                "target": target_company,
                "focus": "TAM/SAM/SOM, competitive landscape, customer concentration, churn",
                "sector": "Semiconductors",
            },
            "timeout": 120,
        },

        # ── Synthesis: waits for ALL three research branches ──────────────
        {
            "id": "synthesis",
            "agent": "EchoAgent",
            # Edge deps: financial → synthesis, legal → synthesis, market → synthesis
            # Explicit deps added for robustness (idempotent with edge deps)
            "depends_on": ["financial_research", "legal_research", "market_research"],
            "params": {
                "task": "Cross-domain synthesis",
                "target": target_company,
                "instruction": (
                    "Synthesise findings from financial, legal, and market research. "
                    "Identify key risks, value drivers, and integration considerations."
                ),
            },
            "timeout": 180,
            "reflection_threshold": 0.0,   # EchoAgent always returns 0.5 → no reflection
        },

        # ── Final report ─────────────────────────────────────────────────
        {
            "id": "final_report",
            "agent": "EchoAgent",
            "params": {
                "task": "Generate executive report",
                "format": "pdf",
                "recipients": ["deal-team@acquirer.com", "board@acquirer.com"],
                "include_epistemic_flags": True,
            },
            "timeout": 60,
        },
    ],

    "edges": [
        # kickoff fans out to all three research nodes
        {"source": "kickoff",            "target": "financial_research"},
        {"source": "kickoff",            "target": "legal_research"},
        {"source": "kickoff",            "target": "market_research"},
        # each research branch feeds synthesis
        {"source": "financial_research", "target": "synthesis"},
        {"source": "legal_research",     "target": "synthesis"},
        {"source": "market_research",    "target": "synthesis"},
        # synthesis feeds final report
        {"source": "synthesis",          "target": "final_report"},
    ],

    "entry": "kickoff",
    "max_cycles": 3,
    "circuit_breaker": {
        "trip_threshold": 3,
        "recovery_executions": 5,
    },
}

print(f"M&A Due Diligence graph for: {target_company}")
print(f"  Nodes : {[n['id'] for n in ma_graph['nodes']]}")
print(f"  Edges : {len(ma_graph['edges'])}")
print(f"  Entry : {ma_graph['entry']}")
print()
print("Expected execution waves:")
print("  Wave 1 → [kickoff]")
print("  Wave 2 → [financial_research, legal_research, market_research]  (parallel BFS)")
print("  Wave 3 → [synthesis]  (gated by depends_on + 3 edge deps)")
print("  Wave 4 → [final_report]")

In [ ]:
# ── Section 7.2: Submit workflow and open SSE stream simultaneously ─────────────

print("Submitting M&A due diligence workflow...")
t_submit_start = time.time()

ma_resp = client.run_graph(
    graph_def=ma_graph,
    payload={
        "initiated_by": "notebook-demo",
        "target_company": target_company,
        "deal_id": "MA-2026-001",
    },
)
ma_wid = ma_resp.instance_id
t_submit_end = time.time()

print(f"Workflow ID    : {ma_wid}")
print(f"Submit latency : {t_submit_end - t_submit_start:.2f}s")
print()
print("Opening SSE stream to watch node completions in real time...")
print()

In [ ]:
# ── Section 7.3: Stream events while workflow runs ─────────────────────────────

ma_events = []

def on_ma_event(ev):
    ma_events.append(ev)
    ts_short = ev['timestamp'].split("T")[1][:12] if ev.get('timestamp') else "?"
    conf = ev.get('confidence', 0)
    print(
        f"  [{ev['index']:02d}]  {ts_short}  "
        f"{ev['node_id']:25s}  "
        f"confidence={conf:.2f}  "
        f"agent={ev.get('agent', '?')}"
    )

print(f"{'idx':>4}  {'time':>12}  {'node':>25}  {'confidence':>10}  agent")
print("-" * 75)

t_workflow_start = time.time()
robust_event_consumer(ma_wid, callback=on_ma_event, max_wait=90)
t_workflow_end = time.time()

print()
print(f"Workflow completed in {t_workflow_end - t_workflow_start:.1f}s")
print(f"Total node completions received: {len(ma_events)}")

In [ ]:
# ── Section 7.4: Verify parallel wave execution ────────────────────────────────

research_nodes = {"financial_research", "legal_research", "market_research"}
research_events = [ev for ev in ma_events if ev["node_id"] in research_nodes]

if len(research_events) == 3:
    from datetime import datetime
    timestamps = [
        datetime.fromisoformat(ev["timestamp"].replace("Z", "+00:00"))
        for ev in research_events
    ]
    max_gap = max(timestamps) - min(timestamps)

    print("Research node completion times (parallel wave):")
    for ev in research_events:
        print(f"  {ev['node_id']:25s}  {ev['timestamp']}")
    print(f"\nMax gap between research nodes: {max_gap.total_seconds():.3f}s")
    print()

    # Verify synthesis ran AFTER all research nodes
    synthesis_events = [ev for ev in ma_events if ev["node_id"] == "synthesis"]
    if synthesis_events:
        syn_idx = synthesis_events[0]["index"]
        research_idxs = [ev["index"] for ev in research_events]
        gated_correctly = all(syn_idx > r_idx for r_idx in research_idxs)
        status = "PASS" if gated_correctly else "FAIL"
        print(f"{status}: synthesis (idx={syn_idx}) ran after all research "
              f"nodes (idxs={research_idxs})")
else:
    print(f"Only {len(research_events)} research events received — workflow may still be running.")
    print("Re-run this cell after the workflow completes.")

In [ ]:
# ── Section 7.5: Fetch the full workflow state ─────────────────────────────────

ma_state = client.get_state(ma_wid)
print(f"Workflow: {ma_wid}")
print(f"Nodes completed: {ma_state.count}")
print()

for node_state in ma_state.nodes:
    output_preview = str(node_state.output)[:80]
    print(f"  {node_state.node_id:25s}  updated={node_state.updated_at}")
    print(f"    output: {output_preview}...")

In [ ]:
# ── Section 7.6: Fetch the epistemic transparency report ──────────────────────
#
# The epistemic report shows per-node confidence, uncertainty propagation,
# known unknowns, and overall trustworthiness. In production this would
# highlight which research branches had low-confidence outputs and flag
# them for human review before the board presentation.

try:
    ep_report = client.get_epistemic_report(ma_wid)
    print("Epistemic Transparency Report")
    print("=" * 50)

    summary = ep_report.get("summary", {})
    print(f"Overall trustworthiness : {summary.get('overall_trustworthiness', 'N/A')}")
    print(f"Mean confidence         : {summary.get('mean_confidence', 'N/A')}")
    print(f"Epistemic debt          : {summary.get('epistemic_debt', 'N/A')}")
    print(f"Flags for human review  : {summary.get('flags', [])}")
    print()

    node_reports = ep_report.get("nodes", [])
    print(f"Per-node breakdown ({len(node_reports)} nodes):")
    for nr in node_reports:
        print(f"  {nr.get('node_id', '?'):25s}  "
              f"confidence={nr.get('confidence', 0):.2f}  "
              f"flags={nr.get('flags', [])}")
except Exception as exc:
    print(f"Epistemic report not available: {exc}")
    print("(The report is generated by the Temporal workflow query handler.")
    print(" It may not be available if the workflow has already completed.")

In [ ]:
# ── Section 7.7: Fetch workflow metrics ───────────────────────────────────────

try:
    metrics = client.get_metrics(ma_wid)
    print("Workflow Execution Metrics")
    print("=" * 40)
    print(f"  nodes_executed         : {metrics.nodes_executed}")
    print(f"  nodes_skipped          : {metrics.nodes_skipped}")
    print(f"  reflections_triggered  : {metrics.reflections_triggered}")
    print(f"  fan_outs_executed      : {metrics.fan_outs_executed}")
    print(f"  circuit_breaker_trips  : {metrics.circuit_breaker_trips}")
    print(f"  error_count            : {metrics.error_count}")
    print(f"  dead_letter_count      : {metrics.dead_letter_count}")
except Exception as exc:
    print(f"Metrics not available: {exc}")

In [ ]:
# ── Section 7.8: Cleanup ───────────────────────────────────────────────────────

client.close()
print("SyncMultigenClient closed.")
print()
print("Workflows executed in this notebook:")
for label, wid in [
    ("Diamond DAG (parallel BFS)",   globals().get('workflow_id', 'not run')),
    ("depends_on (ML inference)",    globals().get('wid2', 'not run')),
    ("Fan-out partition demo",       globals().get('wid3', 'not run')),
    ("SSE stream demo",              globals().get('wid4', 'not run')),
    ("M&A due diligence (full e2e)", globals().get('ma_wid', 'not run')),
]:
    print(f"  {label:35s}  {wid}")

---
## Summary

### What We Demonstrated

| Feature | Where Used | Key API |
|---|---|---|
| **Parallel BFS** | Section 1 — Diamond DAG | `asyncio.gather()` inside `GraphWorkflow.run()` |
| **`depends_on`** | Section 2 — ML inference | `nd.get("depends_on")` in `_deps_satisfied()` |
| **Partition fan-out** | Section 3 — Expert panel | `fan_out_spec["task_queues"]` → round-robin routing |
| **SSE Streaming** | Sections 4, 7 | `GET /workflows/{id}/stream` |
| **Polling fallback** | Section 4.3 | `GET /workflows/{id}/events?since=N` |
| **A2A server** | Section 5.1–5.2 | `GET /.well-known/agent.json`, `POST /a2a` |
| **A2AClient** | Section 5.4 | `A2AClient.send_task()`, `A2AClient.get_agent_card()` |
| **Agent store** | Section 6 | `store_agent_spec()`, `hydrate_agent_if_missing()` |
| **SyncClient in Jupyter** | All sections | `SyncMultigenClient` (background thread + loop) |

### Performance Impact Summary

For a 5-branch M&A expert panel where each expert takes 30s:

| Mode | Total time |
|---|---|
| Sequential BFS (old) | ~150s (sum) |
| Parallel BFS, single queue | ~30s (max) |
| Parallel BFS + partition fan-out (3 workers) | ~30s (max) — no single-queue saturation |

### Configuration Cheatsheet

```bash
# Enable partition-aware fan-out with 3 worker pools
export TEMPORAL_TASK_QUEUES=flow-task-queue,flow-task-queue-b,flow-task-queue-c

# Start a worker for each queue (in separate terminals)
TEMPORAL_TASK_QUEUE=flow-task-queue   python workers/graph_worker.py
TEMPORAL_TASK_QUEUE=flow-task-queue-b python workers/graph_worker.py
TEMPORAL_TASK_QUEUE=flow-task-queue-c python workers/graph_worker.py
```

### Next Steps

- **Notebook 12**: Human approval gates + epistemic debt thresholds
- **Notebook 13**: Circuit breakers + dead-letter replay in production
- **Notebook 14**: MCP (Model Context Protocol) tool integration